# 3DINO-ViT Transfer Experiment — 3D self-supervised foundation model → Glaucoma

Apply **[3DINO-ViT](https://github.com/AICONSlab/3DINO)** (npj Digital Medicine 2025, AICONSlab):
a ViT-Large self-supervised **3D medical imaging foundation model** (~307M params, pretrained on
~100K multi-organ 3D scans) to glaucoma detection on **Harvard-GF OCT** (streamed from HF).

> License: 3DINO code + weights are **CC BY-NC-ND 4.0** — academic/research use only, no commercial use.

**Method (this notebook):**
1. Load pretrained 3DINO-ViT from Hugging Face (gated — accept terms + `HF_TOKEN` secret).
2. Resize OCT to **112³** (3DINO's native size, patch 16 → 7³ tokens).
3. **Linear probe** — freeze backbone, extract the 1024-d CLS embedding per volume, fit
   logistic regression → quick transfer baseline (compares with our UNet-encoder sweep).
4. **Head finetune** — freeze backbone, train a linear head with augmentation + AMP + cosine +
   early stop; optional full finetune toggle.

Weights gated: https://huggingface.co/AICONSlab/3DINO-ViT (accept terms first), then Colab secrets → `HF_TOKEN`.


In [ ]:
# Light deps only — do NOT `pip install -r requirements.txt` (3DINO pins torch 2.0 / xformers / cuml).
# The model falls back to standard attention when xformers is absent.
!pip -q install omegaconf fvcore iopath torchmetrics
!pip -q install hf-transfer huggingface_hub

# Clone 3DINO — used only for its `dinov2` model code.
!git clone --depth 1 https://github.com/AICONSlab/3DINO.git /content/3DINO 2>/dev/null || (cd /content/3DINO && git pull -q)
print("[3dino] repo ready at /content/3DINO")

# Our repo — reused only for `pipeline.plotting` (train/val/test curves + Drive sync).
!git clone --depth 1 https://github.com/Tqhuyen/glaucoma-thesis.git /content/glaucoma-thesis 2>/dev/null || (cd /content/glaucoma-thesis && git pull -q)
print("[repo] glaucoma-thesis ready at /content/glaucoma-thesis")


In [ ]:
# HF token (Colab Secrets -> HF_TOKEN -> hf_xxx) + Drive for saving results
from google.colab import drive, userdata
import os
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")


In [ ]:
# ====== keep the Colab session alive during long runs ======
# Re-clicks Colab's "connect" button every 60s so an idle browser tab does not
# kill the runtime. Keep this tab open and unfocused is fine — but don't close it.
from google.colab import output

JS = """
setInterval(function(){
  const btn = document.querySelector("colab-connect-button");
  if (btn) btn.click();
}, 60000);
"""
try:
    output.eval_js(JS)
    print("[keepalive] armed — runtime will auto-reconnect while this tab stays open.")
except Exception as e:
    print("[keepalive] not available:", e)


In [ ]:
import sys, os, io, json, time, zipfile
from pathlib import Path
sys.path.insert(0, "/content/3DINO")
sys.path.insert(0, "/content/glaucoma-thesis")
from dinov2.configs import load_and_merge_config_3d
from dinov2.eval.setup import build_model_for_eval

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True
_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if _bf16 else torch.float16
SEED = 42
RESOLUTION = 112            # 3DINO-ViT native size (patch 16 -> 7^3 tokens)
FEAT_BS = 16                # feature-extraction batch (frozen backbone)


def preprocess_volume(x):
    """(B,1,200,200,200) uint8 -> (B,1,112,112,112) in [-1,1] (3DINO style)."""
    x = x.float()
    if x.shape[-1] != RESOLUTION:
        x = F.interpolate(x, size=(RESOLUTION,) * 3, mode="trilinear", align_corners=False)
    b = x.shape[0]
    xf = x.reshape(b, 1, -1)
    lo = torch.quantile(xf, 0.0005, dim=2, keepdim=True).unsqueeze(-1).unsqueeze(-1)
    hi = torch.quantile(xf, 0.9995, dim=2, keepdim=True).unsqueeze(-1).unsqueeze(-1)
    x = (x - lo) / (hi - lo + 1e-6)
    return torch.clip(x * 2 - 1, -1, 1)


In [ ]:
# 3DINO-ViT pretrained weights (GATED repo). Accept terms at
# https://huggingface.co/AICONSlab/3DINO-ViT and set HF_TOKEN in Colab secrets first.
from huggingface_hub import hf_hub_download
import os

try:
    WEIGHTS = hf_hub_download(repo_id="AICONSlab/3DINO-ViT", filename="3dino_vit_weights.pth")
except Exception as e:
    print("[3dino] ERROR downloading gated weights:", e)
    print("  1) Open https://huggingface.co/AICONSlab/3DINO-ViT and click 'Agree and access'")
    print("  2) In Colab: key icon (Secrets) -> add HF_TOKEN = hf_xxx")
    print("  3) Re-run the mount cell, then this cell")
    raise
print("[3dino] weights:", WEIGHTS)


In [ ]:
# ================== RAW 200³ data, streamed from Hugging Face ==================
# harvardairobotics/Harvard-GF  (3,300 scans; 2100 / 300 / 900 train/val/test)
HF_REPO  = "harvardairobotics/Harvard-GF"
ZIP_FILE = "Dataset/dataset.zip"       # per-scan .npz with key 'oct_bscans' (200³ uint8)
CSV_FILE = "ReadMe/data_summary.csv"   # columns: filename,glaucoma(yes/no),use(training/validation/test)
DATA_DIR = "/content/glaucoma_hf_200"  # consolidated .npy arrays land here (cached on disk)
SPLITS   = ("Training", "Validation", "Test")

RESOLUTION  = 200                      # RAW storage resolution (never downsampled on disk)
MODEL_RES   = 112                      # 3DINO-ViT native size, applied on-the-fly in preprocess_volume
BATCH_SIZE   = 2                       # 200³ -> tiny per-step batch
GRAD_ACCUM   = 8                       # effective batch = 2 * 8 = 16
CACHE_IN_RAM = False                   # 200³ = ~26 GB total -> mmap, never hold in RAM
NUM_WORKERS  = max(2, os.cpu_count() or 2)

SPLIT_ALIAS = {"training": "Training", "validation": "Validation", "valid": "Validation",
               "test": "Test", "testing": "Test"}


def download_hf(filename):
    from huggingface_hub import hf_hub_download
    print(f"[data] downloading {HF_REPO}/{filename} ...", flush=True)
    return hf_hub_download(repo_id=HF_REPO, filename=filename, repo_type="dataset")


def build_200_data():
    """Stream Harvard-GF -> per-split RAW 200³ .npy arrays (once, low RAM, disk cached)."""
    if all(os.path.isfile(os.path.join(DATA_DIR, f"{s}_volumes.npy")) for s in SPLITS):
        print(f"[data] already built at {DATA_DIR}")
        return
    os.makedirs(DATA_DIR, exist_ok=True)
    csv_path = download_hf(CSV_FILE)
    zip_path = download_hf(ZIP_FILE)

    import csv
    meta = {}
    with open(csv_path, newline="") as fh:
        for r in csv.DictReader(fh):
            split = SPLIT_ALIAS.get((r["use"] or "").strip().lower())
            if split is None:
                continue
            gl = 1 if str(r["glaucoma"]).strip().lower() in ("yes", "1", "true") else 0
            meta[Path(r["filename"]).stem] = (split, gl)
    print(f"[data] {len(meta)} labeled samples from CSV")

    with zipfile.ZipFile(zip_path) as zf:
        names = [n for n in zf.namelist() if n.endswith(".npz")]
        counts = {s: 0 for s in SPLITS}
        for n in names:
            m = meta.get(Path(n).stem)
            if m:
                counts[m[0]] += 1
    print("[data] zip-matched counts:", counts)

    vols, labels = {}, {}
    for s in SPLITS:
        vp = os.path.join(DATA_DIR, f"{s}_volumes.npy")
        vols[s] = np.lib.format.open_memmap(vp, mode="w+", dtype=np.uint8,
                                            shape=(counts[s], 1, RESOLUTION, RESOLUTION, RESOLUTION))
        labels[s] = np.zeros(counts[s], dtype=np.int64)

    filled = {s: 0 for s in SPLITS}
    with zipfile.ZipFile(zip_path) as zf:
        for n in names:
            m = meta.get(Path(n).stem)
            if not m:
                continue
            split, label = m
            raw = np.load(io.BytesIO(zf.read(n)))["oct_bscans"]      # (200,200,200) uint8
            vols[split][filled[split]] = raw[None]                   # -> (1,200,200,200) RAW
            labels[split][filled[split]] = label
            filled[split] += 1
    for s in SPLITS:
        vols[s].flush()
        np.save(os.path.join(DATA_DIR, f"{s}_labels.npy"), labels[s])
        print(f"[data] {s}: {filled[s]} volumes ({(counts[s] * 8 / 1e9):.1f} GB)")
    with open(os.path.join(DATA_DIR, "manifest.json"), "w") as fh:
        json.dump({"source": HF_REPO, "size_name": "200", "store_shape": [1, 200, 200, 200],
                   "splits": {s: {"built_n": filled[s]} for s in SPLITS}}, fh, indent=2)


class OCTMemmapDataset(Dataset):
    """Consolidated {split}_volumes.npy is (N,1,200,200,200) RAW uint8; labels (N,) int64."""

    def __init__(self, data_dir, split, cache_in_ram=False):
        self.labels = np.load(os.path.join(data_dir, f"{split}_labels.npy"))
        vp = os.path.join(data_dir, f"{split}_volumes.npy")
        self.volumes = np.load(vp) if cache_in_ram else np.load(vp, mmap_mode="r")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.from_numpy(np.ascontiguousarray(self.volumes[idx]).copy())  # uint8 (1,200,200,200), writable
        y = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return x, y


def build_loaders():
    build_200_data()
    counts = {s: len(np.load(os.path.join(DATA_DIR, f"{s}_labels.npy"))) for s in SPLITS}
    print("[data] counts:", counts)
    workers = 0 if CACHE_IN_RAM else NUM_WORKERS
    kw = dict(batch_size=BATCH_SIZE, num_workers=workers, pin_memory=(device == "cuda"))
    if workers > 0:
        kw.update(persistent_workers=True, prefetch_factor=4)
    train_ds = OCTMemmapDataset(DATA_DIR, "Training",   cache_in_ram=CACHE_IN_RAM)
    val_ds   = OCTMemmapDataset(DATA_DIR, "Validation", cache_in_ram=CACHE_IN_RAM)
    test_ds  = OCTMemmapDataset(DATA_DIR, "Test",       cache_in_ram=CACHE_IN_RAM)
    g = torch.Generator(); g.manual_seed(SEED)
    train_loader = DataLoader(train_ds, shuffle=True, generator=g, **kw)
    val_loader   = DataLoader(val_ds,   shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  shuffle=False, **kw)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = build_loaders()


In [ ]:
# ====== load pretrained 3DINO-ViT and freeze it ======
cfg = load_and_merge_config_3d("train/vit3d_highres")
model = build_model_for_eval(cfg, WEIGHTS)          # DinoVisionTransformer3d, moved to cuda
for p in model.parameters():
    p.requires_grad_(False)
model.eval()
nparams = sum(p.numel() for p in model.parameters()) / 1e6
print(f"[3dino] loaded {type(model).__name__} | {nparams:.0f}M params | frozen")

# quick sanity forward on one volume (uint8 -> normalized -> feature)
xb, yb = next(iter(test_loader))
with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
    f = model(preprocess_volume(xb[:1].to(device)))
print(f"[3dino] sanity: input {tuple(xb[:1].shape)} -> feature {tuple(f.shape)}")


In [ ]:
# ====== extract frozen 1024-d embeddings for train/val/test (cached on Drive) ======
@torch.no_grad()
def extract_features(loader, out_npz):
    feats, ys = [], []
    for x, y in loader:
        x = preprocess_volume(x.to(device, non_blocking=True))
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            f = model(x)
        feats.append(f.float().cpu())
        ys.append(y)
    Fx = torch.cat(feats)
    Yx = torch.cat(ys)
    np.savez(out_npz, feats=Fx.numpy(), labels=Yx.numpy())
    return Fx, Yx


FEAT_DIR = "/content/drive/MyDrive/MasterBKDN/Thesis/3dino_feats"
os.makedirs(FEAT_DIR, exist_ok=True)


def cached(split, loader):
    p = os.path.join(FEAT_DIR, f"{split}.npz")
    if os.path.exists(p):
        z = np.load(p)
        print(f"[feats] {split}: cached {z['feats'].shape}")
        return torch.from_numpy(np.array(z["feats"], copy=True)), torch.from_numpy(np.array(z["labels"], copy=True))
    Fx, Yx = extract_features(loader, p)
    print(f"[feats] {split}: extracted {tuple(Fx.shape)}")
    return Fx, Yx


Xtr, ytr = cached("Training", train_loader)
Xva, yva = cached("Validation", val_loader)
Xte, yte = cached("Test", test_loader)


In [ ]:
# ====== Linear probe on frozen 3DINO-ViT embeddings ======
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(Xtr.numpy())
clf = LogisticRegression(max_iter=2000, C=1.0)
clf.fit(scaler.transform(Xtr.numpy()), ytr.numpy())

probe = {}
for name, X, y in (("train", Xtr, ytr), ("val", Xva, yva), ("test", Xte, yte)):
    acc = clf.score(scaler.transform(X.numpy()), y.numpy())
    probe[name] = acc
    print(f"[linear-probe] {name:8s} acc = {acc:.4f}")


In [ ]:
# ====== head finetune on frozen 3DINO-ViT (+ optional full finetune) ======
FINETUNE_ALL = False        # True: unfreeze the whole backbone (slow, more VRAM)
FINETUNE_EPOCHS = 15
BATCH = 2
GACC = 8                    # effective batch = 16
LR = 3e-4 if not FINETUNE_ALL else 1e-5
WD, PATIENCE = 0.01, 5

from monai.transforms import (Compose, RandFlip, RandRotate, RandScaleIntensity,
                              RandShiftIntensity, RandGaussianNoise)


def make_aug():
    return Compose([
        RandFlip(prob=0.5, spatial_axis=1),
        RandFlip(prob=0.5, spatial_axis=2),
        RandRotate(range_x=0.05, range_y=0.05, range_z=0.05, prob=0.4,
                   mode="bilinear", padding_mode="zeros", keep_size=True),
        RandScaleIntensity(factors=0.10, prob=0.5),
        RandShiftIntensity(offsets=10.0, prob=0.5),
        RandGaussianNoise(prob=0.3, std=5.0),
    ])


class TrainAug:
    def __init__(self, ds, tf):
        self.ds, self.tf = ds, tf
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        x, y = self.ds[i]
        x = torch.as_tensor(self.tf(x)[0])
        if x.ndim == 3:
            x = x.unsqueeze(0)
        return x, y


class DINOHead(nn.Module):
    def __init__(self, backbone, num_classes=2, dropout=0.3):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(1024, num_classes))
    def forward(self, x):
        x = preprocess_volume(x)
        return self.head(self.backbone(x))


net = DINOHead(model, num_classes=2)
if FINETUNE_ALL:
    for p in net.backbone.parameters():
        p.requires_grad_(True)
    print("[finetune] FULL finetune (backbone unfrozen)")
else:
    print("[finetune] head-only finetune (backbone frozen)")

train_ft = DataLoader(TrainAug(train_loader.dataset, make_aug()), batch_size=BATCH,
                      shuffle=True, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED),
                      pin_memory=(device == "cuda"))

crit = nn.CrossEntropyLoss()
params = [p for p in net.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=WD)
import math
steps_ep = math.ceil(len(train_ft) / GACC)
total = steps_ep * FINETUNE_EPOCHS
warmup = int(total * 0.05)
if warmup > 0:
    sched = torch.optim.lr_scheduler.SequentialLR(
        opt,
        [torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.01, total_iters=warmup),
         torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total - warmup)],
        milestones=[warmup])
else:
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total)
scaler = torch.amp.GradScaler("cuda",
            enabled=(USE_AMP and device == "cuda" and amp_dtype == torch.float16))


@torch.no_grad()
def eval_acc(model, loader):
    model.eval(); c = t = 0
    for x, y in loader:
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            logits = model(x.to(device, non_blocking=True))
        c += (logits.argmax(1) == y.to(device)).sum().item(); t += y.numel()
    return c / t


@torch.no_grad()
def eval_loss_acc(model, loader):
    """Returns (loss, acc) so live train/val/test curves include both."""
    model.eval(); c = t = 0; ls = 0.0
    for x, y in loader:
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            logits = model(x.to(device, non_blocking=True))
            loss = crit(logits, y.to(device))
        ls += loss.item() * y.numel()
        c += (logits.argmax(1) == y.to(device)).sum().item(); t += y.numel()
    return ls / t, c / t


FT_DIR = "/content/drive/MyDrive/MasterBKDN/Thesis/3dino_ft"
os.makedirs(FT_DIR, exist_ok=True)
metrics_path = os.path.join(FT_DIR, "metrics.jsonl")
open(metrics_path, "w").close()  # fresh file for this run


def log_row(step, **kw):
    with open(metrics_path, "a") as fh:
        fh.write(json.dumps({"step": int(step), **kw}) + "\n")


best_val = 0.0; bad = 0; t0 = time.time()
for ep in range(FINETUNE_EPOCHS):
    net.train(); opt.zero_grad(set_to_none=True); tr_loss = 0.0; tr_n = 0
    for i, (x, y) in enumerate(train_ft):
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            loss = crit(net(x), y) / GACC
        tr_loss += loss.item() * GACC; tr_n += y.numel()
        scaler.scale(loss).backward()
        if (i + 1) % GACC == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            opt.zero_grad(set_to_none=True)
    vl, va = eval_loss_acc(net, val_loader)
    log_row(ep + 1, **{"train/loss": tr_loss / max(tr_n, 1),
            "val/loss": vl, "val/acc": va})
    print(f"ep{ep+1:02d} val={va:.4f} (best {best_val:.4f}) | {(time.time()-t0)/60:.1f} min")
    if va > best_val:
        best_val, bad = va, 0
        torch.save(net.state_dict(), os.path.join(FT_DIR, "best_3dino_head.pt"))
    else:
        bad += 1
        if bad >= PATIENCE:
            print("[early-stop] no improvement on val")
            break

net.load_state_dict(torch.load(os.path.join(FT_DIR, "best_3dino_head.pt"), map_location="cpu"))
test_loss, test_acc = eval_loss_acc(net, test_loader)
log_row(FINETUNE_EPOCHS + 1, **{"test/loss": test_loss, "test/acc": test_acc})
print(f"[finetune] best_val={best_val:.4f} | test={test_acc:.4f}")
print(f"[saved] best_3dino_head.pt + metrics.jsonl -> {FT_DIR}")


In [ ]:
# ====== summary + save to Drive (+ live train/val/test curve) ======
summary = {
    "method": "3DINO-ViT transfer (CC BY-NC-ND 4.0, academic use)",
    "dataset": f"Harvard-GF RAW 200³ -> 3DINO @ {MODEL_RES}^3",
    "backbone": "3DINO-ViT-L (frozen) -> 1024-d CLS embedding",
    "linear_probe": probe,
    "head_finetune": {
        "best_val": float(best_val), "test": float(test_acc),
        "finetune_all": bool(FINETUNE_ALL), "epochs": int(FINETUNE_EPOCHS),
    },
    "baseline_reference": {"previous_best_200_sweep": {"val": 0.7967, "test": 0.7533, "model": "enc-32-d5"}},
}
print(json.dumps(summary, indent=2))
try:
    out = "/content/drive/MyDrive/MasterBKDN/Thesis/3dino_results.json"
    with open(out, "w") as fh:
        json.dump(summary, fh, indent=2)
    print(f"[saved] 3dino_results.json -> {out}")
except Exception as e:
    print("save skipped:", e)

# Live train/val/test curves + Drive sync (reuses pipeline/plotting.py).
from pipeline.plotting import render_and_sync

FT_PLOT_DIR = "/content/drive/MyDrive/MasterBKDN/Thesis/3dino_ft"
try:
    png = render_and_sync(Path(FT_PLOT_DIR), {"logging": {"plot_curves": True, "drive_sync_dir": ""}}, tag="3dino_curves")
    print(f"[plot] 3dino_curves.png -> {png if png else 'n/a'}")
except Exception as e:
    print("[plot] skipped:", e)


In [ ]:
# Done. Release the GPU immediately.
from google.colab import runtime
runtime.unassign()
